In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = str(project_root / "src")

# Portfolio Environment (Phase 1)

In this phase, the reinforcement learning environment is initialized and validated.

The objectives of this phase are:

1. Load the prepared environment dataset.
2. Initialize the Gymnasium environment.
3. Reset the environment.
4. Verify the observation shape.
5. Inspect the generated state before implementing the interaction logic.

At this stage, the environment only provides observations. No actions, rewards or transitions are implemented yet.

In [2]:
from environment import PortfolioEnv
from feature_engineering import FeatureEngineer
from feature_scaler import FeatureScaler

In [3]:
engineer = FeatureEngineer()
scaler = FeatureScaler()

environment_data = engineer.load_dataset()
scaled_data = scaler.fit_transform(environment_data)

In [4]:
env = PortfolioEnv(
    dataset=scaled_data,
    price_data=environment_data,
    portfolio="Conservative",
)

In [5]:
state, info = env.reset()

In [6]:
print(state.shape)

(16, 10)


In [7]:
state

array([[-6.1758709e-01, -2.7469486e-01, -5.7064652e-01, -1.9004434e-01,
        -5.4081631e-01, -3.8533962e-01, -2.3580652e-01, -5.9199071e-01,
        -1.6817558e-01, -5.8693933e-01],
       [-2.7469486e-01, -2.5102860e-01, -3.8896739e-01,  4.3910807e-01,
        -4.5670521e-01, -1.5998249e-01,  1.4396824e-02, -4.6664560e-01,
         1.6968895e-02, -3.5821021e-01],
       [-5.7064652e-01, -3.8896739e-01, -7.1847683e-01, -3.2462102e-01,
        -5.7048917e-01, -4.7793770e-01, -4.4978192e-01, -5.7980359e-01,
        -4.1451922e-01, -5.5646020e-01],
       [-1.9004434e-01,  4.3910807e-01, -3.2462102e-01,  4.6909105e-02,
        -2.8295290e-01, -3.3351550e-01, -9.6156672e-02, -3.0366597e-01,
         2.4566185e-02, -3.3708867e-01],
       [-5.4081631e-01, -4.5670521e-01, -5.7048917e-01, -2.8295290e-01,
        -7.6969987e-01, -4.6222329e-01, -4.0490440e-01, -6.1176378e-01,
        -4.5099980e-01, -5.3364164e-01],
       [-3.8533962e-01, -1.5998249e-01, -4.7793770e-01, -3.3351550e-01,
   

In [8]:
env.render()

Portfolio : Conservative
Step      : 0
Date      : 2011-01-04 00:00:00


# Portfolio Environment (Phase 2)

In this phase, the interaction logic of the reinforcement learning environment is validated.

The objectives of this phase are:

1. Execute random portfolio allocation actions.
2. Compute daily asset returns.
3. Compute portfolio return.
4. Update portfolio value.
5. Verify reward calculation.
6. Verify episode progression.

At this stage, transaction costs are intentionally ignored. The reward is equal to the daily portfolio return.

In [9]:
import numpy as np

In [10]:
state, info = env.reset()

In [11]:
action = np.ones(10)

action /= action.sum()

In [12]:
next_state, reward, terminated, truncated, info = env.step(action)

print("Reward :", reward)
print("Portfolio Value :", info["portfolio_value"])
print("Date :", info["date"])

Reward : 0.0012603188694920394
Portfolio Value : 1.001260318869492
Date : 2011-01-05 00:00:00


In [13]:
env.render()

Portfolio : Conservative
Step      : 1
Date      : 2011-01-05 00:00:00


# Reward Function Validation (Phase 3)

This phase validates the complete reward mechanism of the reinforcement learning environment.

The objectives are:

1. Execute portfolio allocation actions.
2. Compute daily portfolio return.
3. Compute transaction cost.
4. Compute the final reward.
5. Update portfolio value.
6. Verify that transaction costs are correctly applied whenever portfolio weights change.

In [14]:
import numpy as np

In [15]:
state, info = env.reset()

In [16]:
action = np.array(
    [
        0.30,
        0.10,
        0.10,
        0.10,
        0.10,
        0.05,
        0.05,
        0.05,
        0.10,
        0.05,
    ],
    dtype=np.float32,
)

In [17]:
next_state, reward, terminated, truncated, info = env.step(action)

print(f"Reward            : {reward:.6f}")

print(f"Portfolio Return  : {info['portfolio_return']:.6f}")

print(f"Transaction Cost  : {info['transaction_cost']:.6f}")

print(f"Portfolio Value   : {info['portfolio_value']:.6f}")

Reward            : 0.001110
Portfolio Return  : 0.001610
Transaction Cost  : 0.000500
Portfolio Value   : 1.001110


# Environment Validation (Phase 4)

This phase verifies that the custom Gymnasium environment complies with the Stable-Baselines3 API.

The environment checker validates:

- Observation Space
- Action Space
- Reset API
- Step API
- Returned data types
- Episode termination logic

Passing this validation ensures that the environment is fully compatible with Stable-Baselines3 before training the PPO agent.

In [18]:
from stable_baselines3.common.env_checker import check_env

In [19]:
check_env(env)

f:\SBU\RL\DRL_Final_Project\.venv\Lib\site-packages\stable_baselines3\common\env_checker.py:324: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
f:\SBU\RL\DRL_Final_Project\.venv\Lib\site-packages\stable_baselines3\common\env_checker.py:515: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(
